## Example 1: RichPdb Debugger

In [ ]:
import inspect
import pdb

import pdbr

# Example 1: RichPdb debugger
print("Example 1 - RichPdb")

# RichPdb is pdb with rich output. When IPython is installed its base class is IPython's
# terminal debugger, itself a pdb.Pdb subclass; otherwise it is pdb.Pdb directly.
bases = [b.__name__ for b in pdbr.RichPdb.__mro__[1:]]
print(f"RichPdb base classes: {bases}")
assert inspect.isclass(pdbr.RichPdb) and issubclass(pdbr.RichPdb, pdb.Pdb), bases

entry_points = ["set_trace", "pm", "post_mortem", "run", "pdbr_context", "apdbr_context"]
for name in entry_points:
    assert callable(getattr(pdbr, name)), name
print(f"Callable entry points: {', '.join(entry_points)}")

print("Example 1 completed")

## Example 2: Driving the Debugger with runcall

In [ ]:
import io
import pdb

from pdbr.utils import debugger_cls

# Example 2: drive the debugger with scripted commands
print("Example 2 - runcall")

# pdbr.RichPdb is built on IPython's terminal debugger, which needs a real terminal and cannot be
# created inside a Jupyter kernel. debugger_cls() builds the same rich debugger on the
# standard-library pdb.Pdb, which reads its commands from any file object.
RichStdlibPdb = debugger_cls(klass=pdb.Pdb)
assert issubclass(RichStdlibPdb, pdb.Pdb)

seen = []

def sample_function(x, y):
    result = x + y
    return result * 2

# The debugger stops on the first line of sample_function; the first command records the
# arguments from inside that frame, the second runs on to the end.
commands = io.StringIO("!seen.append((x, y))\ncontinue\n")
dbg = RichStdlibPdb(stdin=commands, stdout=io.StringIO(), nosigint=True, readrc=False)
output = dbg.runcall(sample_function, 3, 4)
print(f"runcall result: {output}, arguments seen in the stopped frame: {seen}")
assert output == 14, output
assert seen == [(3, 4)], seen

print("Example 2 completed")

## Example 3: Post Mortem Debugging

In [ ]:
import io
import pdb

from pdbr.utils import debugger_cls

# Example 3: Post mortem
print("Example 3 - Post mortem")

RichStdlibPdb = debugger_cls(klass=pdb.Pdb)
seen = []

def buggy_function():
    data = [1, 2, 3]
    return data[10]

try:
    buggy_function()
except IndexError as exc:
    tb = exc.__traceback__
else:
    raise AssertionError("buggy_function did not raise IndexError")

# pdbr.post_mortem() opens RichPdb on a traceback the same way: reset(), then interaction().
# Here the commands come from a string: read a local of the frame that raised, then leave.
commands = io.StringIO("!seen.append(list(data))\ncontinue\n")
dbg = RichStdlibPdb(stdin=commands, stdout=io.StringIO(), nosigint=True, readrc=False)
dbg.reset()
dbg.interaction(None, tb)
print(f"data in the failing frame: {seen}")
assert seen == [[1, 2, 3]], seen

print("Example 3 completed")

## Example 4: Context Managers

In [ ]:
import asyncio
import inspect
from concurrent.futures import ThreadPoolExecutor

import pdbr

# Example 4: Context managers
print("Example 4 - Context managers")

# pdbr_context opens pdbr's post-mortem debugger when its block raises. debug=False turns the
# debugger off, as an unattended run needs; the exception then propagates as usual.
with pdbr.pdbr_context(debug=False):
    value = 10 / 2
assert value == 5.0

try:
    with pdbr.pdbr_context(debug=False):
        value = 10 / 0
except ZeroDivisionError:
    print("debug=False: the exception propagated and no debugger started")
else:
    raise AssertionError("pdbr_context(debug=False) swallowed the exception")

# Both context managers also work as decorators; apdbr_context wraps coroutine functions.
@pdbr.pdbr_context(debug=False)
def safe_division(a, b):
    return a / b

@pdbr.apdbr_context(debug=False)
async def async_double(x):
    await asyncio.sleep(0)
    return x * 2

assert safe_division(10, 2) == 5.0
assert inspect.iscoroutinefunction(async_double)
# The notebook kernel already runs an event loop, so run the coroutine on a worker thread.
with ThreadPoolExecutor(max_workers=1) as pool:
    doubled = pool.submit(asyncio.run, async_double(21)).result()
print(f"safe_division(10, 2) = {safe_division(10, 2)}, async_double(21) = {doubled}")
assert doubled == 42, doubled

print("Example 4 completed")

## Example 5: Public API and run()

In [ ]:
import io
import pdb

import pdbr
from pdbr.utils import debugger_cls

# Example 5: Public API and run()
print("Example 5 - Public API and run()")

public = set(pdbr.__all__)
print(f"pdbr.__all__: {sorted(public)}")
expected = {"RichPdb", "set_trace", "pm", "post_mortem", "run", "pdbr_context", "apdbr_context"}
assert expected <= public, expected - public

# pdbr.run(statement) executes a statement under RichPdb. The same on the stdlib-based class,
# with a scripted "continue":
namespace = {}
dbg = debugger_cls(klass=pdb.Pdb)(stdin=io.StringIO("continue\n"), stdout=io.StringIO(),
                                  nosigint=True, readrc=False)
dbg.run("total = sum(range(5))", namespace)
print(f"total = {namespace['total']}")
assert namespace["total"] == 10, namespace

print("Example 5 completed")